In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [ ]:
import yaml
import os

path_aa = "/kaggle/input/datasets/abcxyzi/raddet-icassp-2025/NISTSpecMaxHold256Data.tar.part-aa"
path_ab = "/kaggle/input/datasets/abcxyzi/raddet-icassp-2025/NISTSpecMaxHold256Data.tar.part-ab"

!mkdir -p /kaggle/working/nist256_data
print("Combining and Extracting...")
!cat "{path_aa}" "{path_ab}" > /kaggle/working/nist256_full.tar
!tar -xf /kaggle/working/nist256_full.tar -C /kaggle/working/nist256_data/
!rm /kaggle/working/nist256_full.tar

base_path = '/kaggle/working/nist256_data/NISTSpecMaxHold256Data'
config = {
    'path': base_path,
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'P0N#1', 1: 'P0N#2', 2: 'Q3N#1', 3: 'Q3N#2', 4: 'Q3N#3'}
}

with open('/kaggle/working/final_pynq_data.yaml', 'w') as f:
    yaml.dump(config, f)

print("Setup Complete. You can now run the Ablation cells.")

Combining and Extracting...
Setup Complete. You can now run the Ablation cells.


In [ ]:
import torch
from ultralytics import YOLO


model_path = "/kaggle/input/datasets/ayush34612/best-pt/best.pt"
model = YOLO(model_path)


def hook_fn(module, input, output):
    return output * 0

handle = model.model.model[21].register_forward_hook(hook_fn)

print("--- Running Validation with P5 Features Zeroed Out ---")

results_no_p5 = model.val(data="/kaggle/working/final_pynq_data.yaml", imgsz=256)

handle.remove()

print(f"Original mAP50: [Check your previous training logs]")
print(f"Ablated mAP50 (No P5): {results_no_p5.box.map50}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
--- Running Validation with P5 Features Zeroed Out ---
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2068.5±618.2 MB/s, size: 120.2 KB)
val: Scanning /kaggle/working/nist256_data/NISTSpecMaxHold256Data/labels/val... 3027 images, 2973 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6000/6000 1.6Kit/s 3.8s
val: New cache created: /kaggle/working/nist256_data/NISTSpecMaxHold256Data/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 375/375 18.7it/s 20.1

In [ ]:
from ultralytics import YOLO
import cv2
import os


model = YOLO("/kaggle/input/datasets/ayush34612/best-pt/best.pt")

def is_small(box, img_area, thresh=0.01):
    x1, y1, x2, y2 = box
    area = (x2 - x1) * (y2 - y1)
    return (area / img_area) < thresh


val_images_path = "/kaggle/working/nist256_data/NISTSpecMaxHold256Data/images/val"
image_files = [os.path.join(val_images_path, f) for f in os.listdir(val_images_path)[:50]]

total_detections = 0
small_detections = 0

print("--- Analyzing Small Object Sensitivity ---")
results = model.predict(source=image_files, conf=0.25, save=False)

for r in results:
    h, w = r.orig_img.shape[:2]
    img_area = h * w
    boxes = r.boxes.xyxy.cpu().numpy()
    
    total_detections += len(boxes)
    for b in boxes:
        if is_small(b, img_area):
            small_detections += 1

print(f"Total Detections: {total_detections}")
print(f"Small Detections (<1% area): {small_detections}")
if total_detections > 0:
    print(f"Percentage of Small Objects: {(small_detections/total_detections)*100:.2f}%")

--- Analyzing Small Object Sensitivity ---

0: 256x256 1 P0N#2, 0.8ms
1: 256x256 (no detections), 0.8ms
2: 256x256 (no detections), 0.8ms
3: 256x256 1 Q3N#2, 0.8ms
4: 256x256 (no detections), 0.8ms
5: 256x256 (no detections), 0.8ms
6: 256x256 1 P0N#1, 0.8ms
7: 256x256 (no detections), 0.8ms
8: 256x256 1 P0N#1, 0.8ms
9: 256x256 1 Q3N#3, 0.8ms
10: 256x256 (no detections), 0.8ms
11: 256x256 (no detections), 0.8ms
12: 256x256 1 P0N#2, 0.8ms
13: 256x256 (no detections), 0.8ms
14: 256x256 (no detections), 0.8ms
15: 256x256 (no detections), 0.8ms
16: 256x256 1 Q3N#1, 0.8ms
17: 256x256 (no detections), 0.8ms
18: 256x256 (no detections), 0.8ms
19: 256x256 1 P0N#2, 0.8ms
20: 256x256 1 P0N#1, 0.8ms
21: 256x256 1 Q3N#3, 0.8ms
22: 256x256 1 P0N#1, 0.8ms
23: 256x256 1 P0N#2, 0.8ms
24: 256x256 1 Q3N#3, 0.8ms
25: 256x256 (no detections), 0.8ms
26: 256x256 1 P0N#1, 0.8ms
27: 256x256 (no detections), 0.8ms
28: 256x256 (no detections), 0.8ms
29: 256x256 1 Q3N#2, 0.8ms
30: 256x256 (no detections), 0.8ms
3

In [ ]:
import numpy as np
import cv2
from ultralytics import YOLO
import os

model = YOLO("/kaggle/input/datasets/ayush34612/best-pt/best.pt")

def add_gaussian_noise(image, sigma=25):
    noise = np.random.normal(0, sigma, image.shape).astype(np.float32)
    noisy_img = image.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)


val_path = "/kaggle/working/nist256_data/NISTSpecMaxHold256Data/images/val"
image_files = [os.path.join(val_path, f) for f in os.listdir(val_path)[:20]] 

print(f"--- Running Noise Robustness Test (Sigma={25}) ---")

for img_path in image_files:
    img = cv2.imread(img_path)
    noisy_img = add_gaussian_noise(img, sigma=30) 
    
   
    results = model.predict(source=noisy_img, conf=0.25, save=True, name="noise_test")

print("Done! Check 'runs/detect/noise_test' to see how the model handled the noise.")

--- Running Noise Robustness Test (Sigma=25) ---

0: 256x256 1 P0N#2, 5.1ms
Speed: 0.4ms preprocess, 5.1ms inference, 1.1ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /kaggle/working/runs/detect/noise_test

0: 256x256 (no detections), 5.5ms
Speed: 0.4ms preprocess, 5.5ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /kaggle/working/runs/detect/noise_test-2

0: 256x256 (no detections), 5.7ms
Speed: 0.6ms preprocess, 5.7ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /kaggle/working/runs/detect/noise_test-3

0: 256x256 1 Q3N#2, 5.4ms
Speed: 0.4ms preprocess, 5.4ms inference, 1.1ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /kaggle/working/runs/detect/noise_test-4

0: 256x256 (no detections), 5.5ms
Speed: 0.5ms preprocess, 5.5ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /kaggle/working/runs/detect/noise_test-5

0: 256x256 (no detectio

In [ ]:
from ultralytics import YOLO

model_frozen = YOLO('yolov8s.pt')

print("--- Starting Backbone Frozen Ablation (Short Run) ---")
results = model_frozen.train(
    data='/kaggle/working/final_pynq_data.yaml',
    epochs=10,         
    imgsz=256,
    batch=16,
    freeze=10,         
    device=0,
    name='v8s_frozen_comparison'
)

print("Training complete. Compare the mAP of this run with your best.pt results.")

--- Starting Backbone Frozen Ablation (Short Run) ---
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_pynq_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v8s_frozen_comparison, nbs=64, nms=False, opse

In [ ]:
import pandas as pd

data = {
    "Ablation_Type": ["Baseline", "No P5 Layer", "Frozen Backbone", "Small Object Ratio"],
    "Metric_mAP50": [0.938, 0.937, 0.903, "22.7%"],
    "Status": ["Success", "Robust", "Sensitive", "High Density"]
}

df = pd.DataFrame(data)
df.to_csv('/kaggle/working/ablation_results_summary.csv', index=False)
print("Summary CSV saved!")

Summary CSV saved!
